In [1]:
import os
import time
import pickle

import pandas as pd
import numpy as np
import torch

import requests
import json
from bs4 import BeautifulSoup
from urllib.parse import quote
from lxml import etree

MESH_URL = "https://id.nlm.nih.gov/mesh/sparql" 

In [2]:
with open("../data/01-result/diseases_df.pkl","rb") as f:
    raw_diseases_df = pickle.load(f)
with open("../data/01-result/indications_df.pkl","rb") as f:
    indications_df = pickle.load(f)

indications_df["efo_id"] = indications_df["efo_id"].str.replace(":","_")

# Ontology descriptors

In [4]:
mesh_query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
PREFIX mesh: <http://id.nlm.nih.gov/mesh/>
PREFIX mesh2025: <http://id.nlm.nih.gov/mesh/2025/>
PREFIX mesh2024: <http://id.nlm.nih.gov/mesh/2024/>
PREFIX mesh2023: <http://id.nlm.nih.gov/mesh/2023/>

SELECT ?dis ?tn (GROUP_CONCAT(DISTINCT ?other_tn; separator=",") AS ?all_tn_list)
FROM <http://id.nlm.nih.gov/mesh>
WHERE {  
  ?root rdfs:label "Immune System Diseases"@en .
  ?root meshv:treeNumber ?tn_root .
  ?dis meshv:treeNumber ?tn .
  FILTER(STRSTARTS(STR(?tn),STR(?tn_root))) .
  ?dis meshv:treeNumber ?other_tn .
}
GROUP BY ?dis ?tn
"""

response = requests.get(f"{MESH_URL}?query={quote(mesh_query)}")

if not response.ok:
    response.raise_for_status()


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
root = etree.fromstring(response.text)
ns = {"sr": "http://www.w3.org/2005/sparql-results#"}
mesh_ids = root.xpath("//sr:binding[@name=\"dis\"]/sr:uri/text()", namespaces=ns)
tree_nb = root.xpath("//sr:result/sr:binding[@name=\"tn\"]/sr:uri/text()", namespaces=ns)
all_tree_nb = root.xpath("//sr:result/sr:binding[@name=\"all_tn_list\"]/sr:literal/text()", namespaces=ns)
for ids in tree_nb, mesh_ids:
    ids[:] = [id.split("/")[-1] for id in ids]

all_tree_nb = [ [ tn.split("/")[-1] for tn in tns.split(",")] for tns in all_tree_nb ]


In [ ]:
diseases_df = pd.DataFrame({"mesh_id": mesh_ids,
                            "tree_number": tree_nb,
                            "all_tree_numbers": all_tree_nb})
diseases_df["ontology_depth"] = diseases_df.tree_number.str.split(".").str.len()

descendant_counts = {tn: diseases_df["tree_number"].str.startswith(tn).sum() for tn in diseases_df["tree_number"]}
diseases_df["nb_descentants"] = diseases_df["tree_number"].map(descendant_counts)

In [ ]:
mesh_query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
PREFIX mesh: <http://id.nlm.nih.gov/mesh/>
PREFIX mesh2025: <http://id.nlm.nih.gov/mesh/2025/>
PREFIX mesh2024: <http://id.nlm.nih.gov/mesh/2024/>
PREFIX mesh2023: <http://id.nlm.nih.gov/mesh/2023/>

SELECT ?tn ?lab
FROM <http://id.nlm.nih.gov/mesh>
WHERE {  
?cat rdfs:label ?lab .
?cat meshv:treeNumber ?tn .
 FILTER(STRSTARTS(STR(?tn), "http://id.nlm.nih.gov/mesh/C") && STRLEN(STR(?tn)) < 31 ).
}
"""

response = requests.get(f"{MESH_URL}?query={quote(mesh_query)}")

if not response.ok:
    response.raise_for_status()

root = etree.fromstring(response.text)
ns = {"sr": "http://www.w3.org/2005/sparql-results#"}
category_ids = root.xpath("//sr:binding[@name=\"tn\"]/sr:uri/text()", namespaces=ns)
category_names = root.xpath("//sr:binding[@name=\"lab\"]/sr:literal/text()", namespaces=ns)

category_ids = [id.split("/")[-1] for id in category_ids]
category_names = [name.lower().replace(" ", "_").replace(",", "") for name in category_names]

categories = dict(zip(category_ids, category_names))

In [ ]:
diseases_df["categories"] = [ [ categories.get(tn.split(".")[0], "not_disease") for tn in tns ] for tns in  diseases_df["all_tree_numbers"] ]

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
encoded_cat = pd.DataFrame(mlb.fit_transform(diseases_df["categories"]), 
                           columns= [f"cat_{c}" for c in mlb.classes_])
diseases_df = pd.concat([diseases_df.drop(columns=["tree_number", "all_tree_numbers", "categories"]), 
                         encoded_cat], axis=1)

# Prevalence proxies

In [ ]:
mesh2efo = dict(zip(indications_df["mesh_id"],indications_df["efo_id"]))
diseases_df["disease_id"] = diseases_df["mesh_id"].apply(lambda id : mesh2efo.get(id, ""))

nb_indications = {mesh:(indications_df["efo_id"] == mesh2efo.get(mesh, "")).sum() for mesh in mesh_ids}
nb_indications
diseases_df["nb_indications"] = diseases_df["mesh_id"].map(nb_indications)

# Save results

In [ ]:
diseases_df = diseases_df.drop(columns=["mesh_id"])
with open("../data/01-result/extended_diseases_df.pkl","wb") as f:
    pickle.dump(raw_diseases_df.merge(diseases_df, on="disease_id"),f)